# GPA778 - Laboratoire 2
# Partie C - Perception III : Actions ROS2 et Filtrage de Nuages de Points

Lors du dernier laboratoire, vous avez implémenté un système de scan 3D avec un LiDAR 2D monté sur servo, contrôlé par des **services ROS2**. Aujourd'hui, vous allez découvrir les **actions ROS2**, un mécanisme plus adapté aux opérations longues, et apprendre à filtrer les nuages de points.

**Objectifs du laboratoire :**
- Comprendre la différence entre services et actions ROS2
- Comprendre le rôle des executors et des callback groups ROS2
- Intégrer une machine à états dans le callback d'exécution d'une action
- Intégrer un état de traitement dans la machine à états
- Implémenter des filtres de nuages de points avec Open3D
- Configurer les paramètres de filtrage via une action ROS2

**Durée estimée :** 2h30

---
## Section 1 : Rappel des Services ROS2

### Qu'est-ce qu'un service ROS2 ?

Un service ROS2 est un mécanisme de communication **synchrone** de type **requête-réponse**.

**Caractéristiques :**
- Le client envoie une requête et **attend** la réponse
- Bloquant pour le client pendant toute la durée d'exécution
- Adapté pour des opérations **courtes** (<1 seconde)

**Exemple du Lab 2b : SetServoPos**

Vous avez utilisé le service `SetServoPos` pour contrôler la position du servo :

```python
# Créer le client
self.set_servo_pos_srv = self.create_client(SetServoPos, 'set_servo_pos')

# Envoyer une requête
req = SetServoPos.Request()
req.position = 2000
future = self.set_servo_pos_srv.call_async(req)
future.add_done_callback(self._on_servo_response)
```

### Limitations des services

**Question de réflexion :**
> "Que se passe-t-il si un service prend 10 secondes à s'exécuter ? Le client peut-il savoir où en est l'exécution ?"

**Réponse :** Non ! Le client attend en silence jusqu'à recevoir la réponse finale. Il n'a **aucun feedback** pendant l'exécution.

---
## Section 2 : Introduction aux Actions ROS2

### Le problème avec les services pour les opérations longues

Imaginez un scan 3D qui prend **8 secondes** à s'exécuter :
- Le client est **bloqué** pendant 8 secondes
- Impossible de savoir si le scan est à 10%, 50% ou 90%
- Impossible d'**annuler** le scan en cours
- Impossible de savoir si le serveur est planté ou simplement lent

**Solution : Les Actions ROS2**

### Qu'est-ce qu'une Action ROS2 ?

Une action est un mécanisme de communication **asynchrone** pour les opérations **longues** avec :
- **Feedback périodique** pendant l'exécution
- Possibilité d'**annulation** (cancel)
- Résultat final à la fin

### Comparaison Service vs Action

| Aspect | Service | Action |
|--------|---------|--------|
| Durée | Courte (<1s) | Longue (>1s) |
| Feedback | ❌ Non | ✅ Oui (périodique) |
| Annulation | ❌ Non | ✅ Oui |
| Résultat | Immédiat | Différé |
| Blocage client | ✅ Bloquant | ❌ Asynchrone |
| Exemple | `SetServoPos` | `StartScan` |

### Structure d'une action

Une action ROS2 contient **trois sections** :

#### 1. **Goal** (Objectif)
Les paramètres de l'opération à effectuer (équivalent à la requête d'un service)

#### 2. **Result** (Résultat)
Le résultat final renvoyé à la fin (équivalent à la réponse d'un service)

#### 3. **Feedback** (Retour)
Des informations envoyées **périodiquement** pendant l'exécution (ex: pourcentage de progression)

### Exemple : Fichier StartScan.action

**Exercice de lecture :**
> "Ouvrez le fichier `/home/ubuntu/ros2_ws/src/jerro/jerro_msgs/action/StartScan.action` et identifiez les trois sections."

In [ ]:
# Visualisons la structure de StartScan.action
!cat /home/ubuntu/ros2_ws/src/jerro/jerro_msgs/action/StartScan.action

### Analyse de StartScan.action

**Goal (Objectif) :**
- `start` : booléen pour démarrer le scan
- **Nouveaux paramètres de filtrage** : enable_filtering, enable_statistical_filter, etc.

**Result (Résultat) :**
- `point_cloud` : Le nuage de points 3D résultant
- `success` : Booléen indiquant si le scan a réussi
- `message` : Message descriptif

**Feedback (Retour périodique) :**
- `progress` : Float entre 0.0 et 1.0 indiquant la progression du scan

### Diagramme de séquence

```
Client                    Action Server
  |                             |
  |──────(Goal)───────────────> |
  |                             | goal_callback()
  | <───(Accept/Reject)────────|
  |                             | execute_callback()
  | <───(Feedback 10%)─────────|
  |                             |
  | <───(Feedback 50%)─────────|
  |                             |
  | <───(Feedback 90%)─────────|
  |                             |
  | <───(Result + Success)─────|
  |                             |
```

---
## Section 3 : L'Action StartScan dans lidar_traitement.py
### Création d'un Action Server

Dans le fichier `lidar_traitement.py`, un Action Server est créé ainsi :

```python
from rclpy.action import ActionServer, GoalResponse, CancelResponse
from jerro_msgs.action import StartScan

self._action_server = ActionServer(
    self,                                    # Le node ROS2
    StartScan,                               # Type de l'action
    '/perception/start_scan',                # Nom du topic
    execute_callback=self.execute_scan_callback,
    goal_callback=self.goal_callback,
    cancel_callback=self.cancel_callback,
    callback_group=self._group,              # Voir Section 4
)
```

### Les trois callbacks

#### 1. **goal_callback** : Accepter ou rejeter le goal

```python
def goal_callback(self, goal_request):
    if not self.set_servo_pos_srv.service_is_ready():
        return GoalResponse.REJECT  # Servo pas prêt
    if self.actual_scan is None:
        return GoalResponse.REJECT  # Pas de scan reçu
    return GoalResponse.ACCEPT      # OK, on accepte
```

#### 2. **cancel_callback** : Gérer les demandes d'annulation

```python
def cancel_callback(self, goal_handle):
    self.get_logger().info("Annulation demandée")
    return CancelResponse.ACCEPT
```

#### 3. **execute_callback** : Exécuter l'action avec une boucle

C'est le callback central. Il **démarre quand le goal est accepté** et **se termine quand il retourne un résultat** :

```python
def execute_scan_callback(self, goal_handle):
    # Initialisation...
    self._scan_state.set_next_state(ScanState.MOVE_TO_START)

    while True:                                    # ← boucle principale
        match self._scan_state.get_actual_state():
            case ScanState.FINISHING:
                self._finalize_scan()
                result = StartScan.Result()
                result.success = True
                goal_handle.succeed()
                return result                      # ← fin du callback (succès)

            case ScanState.WAITING_SERVO:
                if prev == ScanState.CANCELLING:
                    goal_handle.canceled()
                    return result                  # ← fin du callback (annulation)
            # ...
```

**Points clés :**
- Le callback **tourne en boucle** tant que le scan n'est pas terminé
- Il se termine par un `return result` (succès ou annulation)
- `goal_handle` est disponible localement (pas besoin de l'stocker comme attribut de classe)
- Le feedback est publié directement depuis la boucle : `goal_handle.publish_feedback(feedback)`

### Questions de compréhension

1. **Quelle ligne termine le callback en cas de succès ?**
   - Réponse : `return result` dans le case `FINISHING`, après `goal_handle.succeed()`

2. **Que se passe-t-il si `goal_callback` retourne `GoalResponse.REJECT` ?**
   - Réponse : `execute_callback` n'est jamais appelé

3. **Comparez avec la v1 : dans `lidar_traitement_v1.py`, qui pilotait la machine à états ?**
   - Réponse : Un timer appelait `state_machine_tick()` toutes les 50ms. Ici, c'est la boucle `while True`.

---
## Section 4 : Executor et Callback Groups

### C'est quoi un Executor ?

Un **executor** est le composant ROS2 qui s'occupe d'**exécuter les callbacks** : timers, subscribers, services, actions...

Quand vous écrivez `rclpy.spin(node)`, vous utilisez implicitement un executor **mono-thread** qui traite tous les callbacks **l'un après l'autre**.

### Le problème : execute_callback est bloquant

`execute_scan_callback` contient une boucle `while True` qui **ne se termine qu'à la fin du scan** (plusieurs secondes). Pendant ce temps, le thread est occupé.

Or, pendant ce `while True`, la boucle attend que `self._servo_done` passe à `True` :

```python
case ScanState.WAITING_SERVO:
    if not self._servo_done:   # ← attend ici (busy-wait)
        continue
```

Qui met `_servo_done = True` ? C'est `_on_servo_response`, un **callback** appelé quand le servo confirme sa position :

```python
def _on_servo_response(self, _future):
    self._servo_done = True     # ← doit s'exécuter pendant le while True !
```

### Le deadlock avec un seul thread

```
Thread unique :
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  1. execute_scan_callback() démarre, entre dans while True  │
│     → envoie commande servo (_send_servo_pos)               │
│     → WAITING_SERVO : boucle en attendant _servo_done...    │
│         while not self._servo_done: continue  ←──BLOQUÉ !  │
│                                                             │
│  2. _on_servo_response() veut s'exécuter...                 │
│     → IMPOSSIBLE ! Le thread est pris par le while True     │
│                                                             │
│  3. _servo_done ne sera JAMAIS mis à True                   │
│     → DEADLOCK                                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### La solution : MultiThreadedExecutor

Avec plusieurs threads, les callbacks peuvent s'exécuter en parallèle :

```
Thread 1 :  execute_scan_callback()  ──[while True... WAITING_SERVO...]──
                                                                         ↑ débloqué !
Thread 2 :                    _on_servo_response()  →  _servo_done = True
Thread 3 :         scan_callback()  (reçoit les données LiDAR en continu)
```

**Implementation :**

```python
from rclpy.executors import MultiThreadedExecutor

def main(args=None):
    rclpy.init(args=args)
    node = Lidar3D()
    executor = MultiThreadedExecutor()   # ← plusieurs threads
    executor.add_node(node)
    executor.spin()                      # ← remplace rclpy.spin()
```

### Les Callback Groups

Mais le `MultiThreadedExecutor` seul ne suffit pas ! ROS2 utilise les **Callback Groups** pour décider quels callbacks peuvent s'exécuter en parallèle.

#### MutuallyExclusiveCallbackGroup (comportement par défaut)

Les callbacks du **même groupe** ne s'exécutent **jamais en parallèle** — un seul à la fois :

```
Groupe A (MutuallyExclusive) :
  callback_1 ──────────────────┐
  callback_2                   └──────────────── (doit attendre)
```

Si `execute_scan_callback` et `cancel_callback` appartiennent au **même** `MutuallyExclusiveCallbackGroup`, l'annulation ne pourra jamais être traitée pendant l'exécution du scan.

#### ReentrantCallbackGroup

Les callbacks du **même groupe** **peuvent** s'exécuter en parallèle :

```
Groupe B (Reentrant) :
  execute_callback ──────────────────────────────────
  cancel_callback       ──────                       (peut s'exécuter en même temps !)
```

**En pratique :**

Chaque callback créé séparément (par `create_subscription`, `create_timer`, etc.) reçoit **son propre** `MutuallyExclusiveCallbackGroup` par défaut. Ces groupes **différents** peuvent donc toujours s'exécuter en parallèle avec `MultiThreadedExecutor`.

Le problème spécifique à notre cas est que **plusieurs callbacks de l'action** (execute + cancel) appartiendraient au même groupe par défaut. En les mettant dans un `ReentrantCallbackGroup`, on garantit que l'annulation peut interrompre l'exécution :

```python
from rclpy.callback_groups import ReentrantCallbackGroup

self._group = ReentrantCallbackGroup()

self._action_server = ActionServer(
    self,
    StartScan,
    '/perception/start_scan',
    execute_callback=self.execute_scan_callback,
    goal_callback=self.goal_callback,
    cancel_callback=self.cancel_callback,
    callback_group=self._group,          # ← Reentrant : exécution en parallèle possible
)
```

### Tableau récapitulatif

| | `rclpy.spin()` | `MultiThreadedExecutor` seul | `MultiThreadedExecutor` + `ReentrantCallbackGroup` |
|-|----------------|------------------------------|------------------------------------------------------|
| **Threads** | 1 | N | N |
| **while True bloquant** | ❌ Deadlock | ✅ OK | ✅ OK |
| **Annulation pendant scan** | ❌ Impossible | ⚠️ Partiel | ✅ Garanti |
| **scan_callback en parallèle** | ❌ | ✅ | ✅ |

### Questions de compréhension

1. **Si on utilise `rclpy.spin()` avec le while True, que se passe-t-il lors de WAITING_SERVO ?**
   - Réponse : Deadlock — le while True monopolise le thread, `_on_servo_response` ne peut jamais s'exécuter, `_servo_done` reste à False pour toujours.

2. **Sans `ReentrantCallbackGroup`, que se passerait-il si on envoie une annulation pendant le scan ?**
   - Réponse : `cancel_callback` ne pourrait pas s'exécuter pendant que `execute_scan_callback` tourne (même groupe `MutuallyExclusive`).

3. **La `lidar_traitement.py` utilisait `rclpy.spin()`. Pourquoi n'avait-elle pas de deadlock ?**
   - Réponse : La machine à états était dans un timer — chaque appel à `state_machine_tick()` se terminait en quelques millisecondes. Pas de boucle bloquante. Le thread était toujours libéré rapidement.

---
## Section 5 : Machine à États dans le while True

### Architecture : timer vs while True

Dans `lidar_traitement_v1.py`, la machine à états était pilotée par un **timer** :

```
Timer (toutes les 50ms) ──> state_machine_tick()  ──> avance d'un état
                     ↑___________________________|  (callback court)
```

Dans `lidar_traitement.py`, elle est intégrée dans la **boucle while de l'action** :

```
execute_scan_callback() ─────────────────────────────────────────────>
   while True:                                                         |
     MOVE_TO_START → WAITING_SERVO → SCANNING → ... → FINISHING        |
                                                         return result |
```

**Avantage :** Toute la logique du scan est dans un seul endroit. Plus lisible, plus direct.

### États actuels de ScanState

```python
class ScanState:
    IDLE = 0          # Pas utilisé dans la boucle (état initial)
    MOVE_TO_START = 1 # Déplacer le servo à la position de départ
    WAITING_SERVO = 2 # Attendre la réponse du servo (busy-wait)
    SCANNING = 3      # Scanner et publier le feedback
    RESETTING = 4     # Remettre le servo à la position neutre
    FINISHING = 5     # Finaliser, publier le nuage, retourner le résultat
    CANCELLING = 6    # Annulation en cours
```

### Diagramme de transitions

```
(goal reçu) → MOVE_TO_START
                   ↓ (_send_servo_pos)
              WAITING_SERVO ←────────────────────────────┐
                   ↓ (_servo_done = True)                │
               SCANNING ──(cancel)──> CANCELLING         │
                   │                      │ (_send_servo_pos(1500))
                   │ (fin de course)       └──────────────┘
                   ↓
               RESETTING
                   ↓ (_send_servo_pos(1500))
              WAITING_SERVO
                   ↓
               FINISHING
                   ↓
             return result
```

### Ajout de l'état PROCESSING

Pour ajouter le filtrage, un état `PROCESSING` doit être inséré entre `WAITING_SERVO` (après RESETTING) et `FINISHING` :

```
... RESETTING
       ↓
  WAITING_SERVO
       ↓ (si enable_filtering)
  PROCESSING          ↓ (sinon)
       └────────────> FINISHING
                          ↓
                    return result
```

**Question :** Comment passer conditionnellement à PROCESSING ou directement à FINISHING ?

**Indice :** Utilisez `goal_handle.request.enable_filtering`

**Attention :** Dans la version avec `while True`, `goal_handle` est disponible comme **variable locale** du callback (pas comme attribut de classe). Vous pouvez l'utiliser directement dans la boucle.

---
## Section 6 : Filtres Open3D - Théorie

### Introduction au filtrage

#### Pourquoi filtrer les nuages de points ?

Les nuages de points bruts issus d'un LiDAR contiennent souvent :
- **Bruit** : Points erronés dus aux limitations du capteur
- **Outliers** : Points aberrants très éloignés de leur voisinage
- **Densité excessive** : Trop de points pour le traitement en temps réel
- **Données hors zone d'intérêt** : Points au-delà de l'horizon immédiat du robot

#### Sources de bruit dans un scan LiDAR

- **Réflexions multiples** : Le laser rebondit plusieurs fois
- **Surfaces réfléchissantes** : Miroirs, vitres
- **Poussière et particules** : Dans l'air
- **Erreurs de mesure** : Imprécision du capteur
- **Vibrations mécaniques** : Du servo ou du robot

### Filtre 1 : Crop (Bounding Box)

#### Principe

Rogne le nuage pour ne garder que les points **à l'intérieur d'un volume défini** (boîte englobante).

Utile pour :
- Limiter la zone d'intérêt (ex: 5m autour du robot)
- Éliminer le sol ou le plafond
- Réduire le volume de données à traiter

#### Paramètres

- `x_min`, `x_max` : Limites en X (avant/arrière du robot)
- `y_min`, `y_max` : Limites en Y (gauche/droite)
- `z_min`, `z_max` : Limites en Z (bas/haut)

#### Visualisation

```
         z ↑
           │    crop_z_max
           │   ┌───────────┐
           │   │           │
           │   │   Zone    │
           │   │ conservée │
           │   │           │
           │   └───────────┘
           │    crop_z_min
           └──────────────────> x
```

#### Méthode Open3D

```python
bbox = o3d.geometry.AxisAlignedBoundingBox(
    min_bound=[x_min, y_min, z_min],
    max_bound=[x_max, y_max, z_max]
)
cropped_cloud = cloud.crop(bbox)
```

#### Référence

**Documentation Open3D :**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

#### Questions

1. **Si le robot fait 0.3m de haut, quelle valeur de `z_min` permet d'éliminer le sol ?**
   - Réponse : Environ `z_min = 0.05` à `0.1` pour éviter de couper le robot lui-même

2. **Pourquoi appliquer le crop en premier ?**
   - Réponse : Pour réduire le volume de données **avant** les autres filtres coûteux

### Filtre 2 : Statistical Outlier Removal

#### Principe

Élimine les points **statistiquement aberrants** basé sur leur distance à leurs voisins.

**Algorithme :**
1. Pour chaque point, calculer la distance moyenne à ses **K voisins** les plus proches
2. Calculer la **moyenne** et l'**écart-type** de toutes ces distances
3. Éliminer les points dont la distance moyenne > `moyenne + std_ratio × écart-type`

#### Visualisation

```
    ●                     ●  ← Outlier isolé (sera supprimé)
    
         ● ● ●
        ● ● ● ●           ← Cluster dense (sera conservé)
         ● ● ●
    
                 ●        ← Outlier (sera supprimé)
```

#### Paramètres

- **`nb_neighbors`** : Nombre de voisins à considérer (ex: 20)
  - Plus élevé → plus robuste mais plus lent
  - Plus faible → plus rapide mais moins fiable

- **`std_ratio`** : Seuil en multiple d'écart-type (ex: 2.0)
  - Plus élevé → **plus permissif** (garde plus de points)
  - Plus faible → **plus strict** (élimine plus de points)

#### Méthode Open3D

```python
filtered_cloud, indices = cloud.remove_statistical_outlier(
    nb_neighbors=20,
    std_ratio=2.0
)
# filtered_cloud : nuage filtré
# indices : liste des indices des points conservés
```

#### Référence

**Documentation Open3D :**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal

#### Questions

1. **Un `std_ratio` plus élevé rendra le filtre plus strict ou plus permissif ?**
   - Réponse : **Plus permissif** (garde plus de points car le seuil est plus haut)

2. **Un `nb_neighbors` trop petit risque d'éliminer des points valides. Pourquoi ?**
   - Réponse : Avec peu de voisins, la distance moyenne est moins fiable statistiquement

3. **Quand utiliser ce filtre ?**
   - Réponse : Pour éliminer le **bruit** et les **points isolés** aberrants

### Filtre 3 : Voxel Downsampling

#### Principe

Réduit la **densité** du nuage de points en utilisant une grille 3D de **voxels** (cubes).

**Algorithme :**
1. Diviser l'espace en grille 3D de voxels de taille fixe
2. Pour chaque voxel contenant des points :
   - Calculer le **centroïde** (moyenne) des points dans le voxel
   - Remplacer tous les points par ce centroïde unique

**Résultat :** Maximum un point par voxel

#### Visualisation 2D

```
Avant :                  Après :
┌────┬────┬────┐        ┌────┬────┬────┐
│ ●● │ ●  │    │        │ ●  │ ●  │    │
│ ●  │    │  ● │   →    │    │    │  ● │
├────┼────┼────┤        ├────┼────┼────┤
│ ●●●│ ●● │    │        │ ●  │ ●  │    │
│ ●  │ ●  │    │        │    │    │    │
└────┴────┴────┘        └────┴────┴────┘

8 points → 4 points
```

#### Paramètres

- **`voxel_size`** : Taille du voxel en mètres (ex: 0.02 = 2 cm)
  - Plus grand → **moins de points** (plus de réduction)
  - Plus petit → **plus de points** (moins de réduction)

#### Méthode Open3D

```python
downsampled_cloud = cloud.voxel_down_sample(voxel_size=0.02)
```

#### Référence

**Documentation Open3D :**  
https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling

#### Questions

1. **Un `voxel_size` de 0.05m au lieu de 0.02m donnera plus ou moins de points ?**
   - Réponse : **Moins de points** (voxels plus grands = moins de voxels = moins de points)

2. **Quel est l'avantage du voxel downsampling par rapport à un simple échantillonnage aléatoire ?**
   - Réponse : Conserve une **distribution uniforme** dans l'espace, pas de zones vides

3. **Quand utiliser ce filtre ?**
   - Réponse : Pour réduire la **densité** et accélérer les traitements ultérieurs

### Ordre d'application des filtres

L'ordre des filtres est **important** pour l'efficacité et la qualité du résultat.

#### Ordre recommandé

1. **Crop** en premier
   - Pourquoi ? Réduit le volume de données **avant** les filtres coûteux
   - Exemple : 100 000 points → 50 000 points

2. **Statistical Outlier Removal** en deuxième
   - Pourquoi ? Élimine le bruit sur des données déjà réduites
   - Exemple : 50 000 points → 45 000 points

3. **Voxel Downsampling** en dernier
   - Pourquoi ? Optimise la densité finale après nettoyage
   - Exemple : 45 000 points → 10 000 points

#### Pipeline de filtrage

```
Nuage brut (100k points, bruité, zone large)
    ↓
[CROP] → Limite à la zone d'intérêt
    ↓
Nuage rogné (50k points, bruité, zone réduite)
    ↓
[STATISTICAL] → Élimine les outliers
    ↓
Nuage nettoyé (45k points, propre, zone réduite)
    ↓
[VOXEL] → Réduit la densité
    ↓
Nuage final (10k points, propre, optimisé)
```

#### Tableau récapitulatif

| Filtre | Objectif | Réduit points ? | Coût calcul |
|--------|----------|-----------------|-------------|
| Crop | Limite la zone | Oui (beaucoup) | Faible |
| Statistical | Élimine le bruit | Oui (modéré) | Élevé |
| Voxel | Réduit densité | Oui (beaucoup) | Modéré |

---
## Section 7 : Exercices Pratiques

### Préparation de l'environnement

Avant de commencer, mettez à jour votre workspace avec la nouvelle définition d'action :

In [ ]:
%%bash
cd ~/ros2_ws/src/jerro
git pull
cd ~/ros2_ws
colcon build --packages-select jerro_msgs
source install/setup.bash

### Exercice 1 : Compréhension de la nouvelle définition d'action

**Objectif :** Analyser la définition modifiée de StartScan.action

**Instructions :**
1. Ouvrez le fichier `StartScan.action`
2. Identifiez tous les nouveaux paramètres ajoutés dans le Goal
3. Pour chaque paramètre, expliquez son rôle en vous aidant de la documentation Open3D

**Questions :**
- Pourquoi certains paramètres sont des `bool` et d'autres des `float32` ?
- Combien de paramètres au total ont été ajoutés ?
- Quelle est la valeur par défaut suggérée pour `voxel_size` ?

### Exercice 2 : Remplacement du timer par le while True (30 min)

**Objectif :** Transformer `execute_scan_callback` pour qu'il contienne directement la machine à états

**Fichier à modifier :** `lidar_traitement.py`

**Contexte :** La version actuelle du fichier contient encore le timer commenté. Votre tâche est de réécrire `execute_scan_callback` avec la boucle `while True`.

**TODO 1 : Configurer le ReentrantCallbackGroup**

```python
# TODO : Importez ReentrantCallbackGroup
from rclpy.callback_groups import ___

# TODO : Créez le groupe dans __init__
self._group = ___

# TODO : Assignez-le à l'action server
self._action_server = ActionServer(
    ...
    callback_group=___,
)
```

**TODO 2 : Réécrire execute_scan_callback**

```python
def execute_scan_callback(self, goal_handle):  # ← plus de async
    # TODO : Initialiser le scan (servo_pos, global_scan_3d, servo_done, état initial)

    while True:
        match self._scan_state.get_actual_state():

            case ScanState.MOVE_TO_START:
                # TODO : Envoyer le servo à la position de départ

            case ScanState.WAITING_SERVO:
                # TODO : Busy-wait si _servo_done est False
                # TODO : Selon l'état précédent, passer à SCANNING, FINISHING ou gérer l'annulation

            case ScanState.SCANNING:
                # TODO : Vérifier si annulation demandée (goal_handle.is_cancel_requested)
                # TODO : Collecter le scan et publier le feedback
                # TODO : Décrémenter la position servo et envoyer si nécessaire

            case ScanState.RESETTING:
                # TODO : Envoyer le servo à la position neutre (1500)

            case ScanState.CANCELLING:
                # TODO : Envoyer le servo à la position neutre

            case ScanState.FINISHING:
                # TODO : Finaliser le scan, créer le result, appeler goal_handle.succeed()
                # TODO : return result  ← termine le callback
```

**TODO 3 : Utiliser MultiThreadedExecutor dans main()**

```python
def main(args=None):
    rclpy.init(args=args)
    node = Lidar3D()
    # TODO : Créer un MultiThreadedExecutor, ajouter le node, appeler spin()
```

**Indices :**
- `goal_handle` est disponible comme variable locale dans `execute_scan_callback`
- Pour le busy-wait : `if not self._servo_done: continue`
- Le callback se termine par `return result` (pas de Future ni de `set_result`)
- N'oubliez pas de réinitialiser `_servo_done = False` avant d'envoyer une commande servo

### Exercice 3 : Ajout de l'état PROCESSING

**Objectif :** Intégrer un nouvel état dans la machine à états

**Fichier à modifier :** `/home/ubuntu/ros2_ws/src/jerro_perception/jerro_perception/lidar_traitement.py`

**TODO 1 : Modifier la classe ScanState**

```python
class ScanState:
    IDLE = 0
    MOVE_TO_START = 1
    WAITING_SERVO = 2
    SCANNING = 3
    RESETTING = 4
    # TODO : Ajoutez l'état PROCESSING = ___
    FINISHING = ___  # TODO : Ajustez la valeur
    CANCELLING = ___  # TODO : Ajustez la valeur
```

**TODO 2 : Ajouter le case PROCESSING dans state_machine_tick()**

```python
case ScanState.PROCESSING:
    # TODO : Appelez la méthode self._process_point_cloud()
    # TODO : Passez à l'état FINISHING
    pass
```

**TODO 3 : Modifier le case WAITING_SERVO pour la transition conditionnelle**

```python
case ScanState.WAITING_SERVO:
    if not self._servo_done:
        return
    prev = self._scan_state.get_previous_state()
    if prev == ScanState.RESETTING:
        # TODO : Vérifier si le filtrage est activé dans le goal
        # TODO : Si oui, passer à PROCESSING, sinon à FINISHING
        pass
```

**Indice :** Utilisez `self._goal_handle.request.enable_filtering`

### Exercice 4 : Implémentation des filtres

**Objectif :** Créer les trois méthodes de filtrage

**Fichier à modifier :** `lidar_traitement.py`

#### Filtre 1 : _apply_crop

```python
def _apply_crop(self, cloud, x_range, y_range, z_range):
    """
    TODO : Rogne le nuage dans une boîte englobante

    Args:
        cloud: Nuage Open3D
        x_range: [min, max] en x
        y_range: [min, max] en y
        z_range: [min, max] en z

    Returns:
        Nuage rogné

    Référence: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

    Indices :
    - Créez un AxisAlignedBoundingBox avec min_bound et max_bound
    - Utilisez la méthode .crop() du nuage
    - Loggez le nombre de points avant/après
    """
    # TODO : Votre code ici
    pass
```

#### Filtre 2 : _apply_statistical_filter

```python
def _apply_statistical_filter(self, cloud, nb_neighbors, std_ratio):
    """
    TODO : Filtre statistique pour éliminer les outliers

    Args:
        cloud: Nuage Open3D
        nb_neighbors: Nombre de voisins à considérer
        std_ratio: Ratio d'écart-type

    Returns:
        Nuage filtré

    Référence: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal

    Indices :
    - Utilisez la méthode .remove_statistical_outlier()
    - Cette méthode retourne DEUX valeurs : (nuage_filtré, indices)
    - Loggez le nombre de points avant/après
    """
    # TODO : Votre code ici
    pass
```

#### Filtre 3 : _apply_voxel_downsampling

```python
def _apply_voxel_downsampling(self, cloud, voxel_size):
    """
    TODO : Sous-échantillonne le nuage avec une grille de voxels

    Args:
        cloud: Nuage Open3D
        voxel_size: Taille des voxels en mètres

    Returns:
        Nuage sous-échantillonné

    Référence: https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling

    Indices :
    - Utilisez la méthode .voxel_down_sample()
    - Loggez le nombre de points avant/après
    """
    # TODO : Votre code ici
    pass
```

### Exercice 5 : Intégration dans _process_point_cloud()

**Objectif :** Créer la méthode qui orchestre tous les filtres

```python
def _process_point_cloud(self):
    """
    TODO : Appliquez les filtres configurés dans le goal sur le nuage global

    Ordre d'application :
    1. Crop (réduit le volume de données)
    2. Statistical outlier removal (élimine le bruit)
    3. Voxel downsampling (optimise la densité finale)

    Chaque filtre ne doit être appliqué que si son booléen d'activation est True
    """
    if self._goal_handle is None:
        return

    goal = self._goal_handle.request
    self.get_logger().info("Début du traitement du nuage de points...")

    filtered_cloud = self.global_scan_3d

    # TODO 1 : Appliquer le crop si goal.enable_crop est True
    # Indices :
    #   - Créez les ranges : x_range = [goal.crop_x_min, goal.crop_x_max]
    #   - Appelez self._apply_crop(filtered_cloud, x_range, y_range, z_range)
    #   - Assignez le résultat à filtered_cloud

    # TODO 2 : Appliquer le filtre statistique si goal.enable_statistical_filter est True
    # Indices :
    #   - Récupérez les paramètres : goal.statistical_nb_neighbors, goal.statistical_std_ratio
    #   - Appelez self._apply_statistical_filter(...)

    # TODO 3 : Appliquer le voxel downsampling si goal.enable_voxel_downsampling est True
    # Indices :
    #   - Récupérez le paramètre : goal.voxel_size
    #   - Appelez self._apply_voxel_downsampling(...)

    # Remplacer le nuage global par le nuage filtré
    self.global_scan_3d = filtered_cloud
    self.get_logger().info("Traitement du nuage de points terminé")
```

### Exercice 6 : Test et ajustement

**Objectif :** Tester votre implémentation et trouver les paramètres optimaux

#### Étape 1 : Compilation

In [ ]:
%%bash
cd ~/ros2_ws
colcon build --packages-select jerro_perception
source install/setup.bash

#### Étape 2 : Lancer le node

```bash
ros2 run jerro_perception lidar_traitement
```

#### Étape 3 : Envoyer un goal avec filtrage activé

**Terminal 2 :**
```bash
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan \
  "{
    start: true,
    enable_filtering: true,
    enable_crop: true,
    crop_x_min: -2.5, crop_x_max: 2.5,
    crop_y_min: -2.5, crop_y_max: 2.5,
    crop_z_min: -0.5, crop_z_max: 1.0,
    enable_statistical_filter: true,
    statistical_nb_neighbors: 20,
    statistical_std_ratio: 2.0,
    enable_voxel_downsampling: true,
    voxel_size: 0.02
  }" \
  --feedback
```

#### Étape 4 : Visualisation dans RViz

**Terminal 3 :**
```bash
rviz2
```

- Ajoutez un display `PointCloud2`
- Topic : `/perception/lidar_pc2`
- Fixed Frame : `base_link`

#### Étape 5 : Expérimentation

Testez différentes configurations :

1. **Sans filtrage** : `enable_filtering: false`
2. **Crop seul** : Seulement enable_crop
3. **Statistical seul** : Seulement enable_statistical_filter
4. **Tous les filtres** : Tous activés
5. **Paramètres agressifs** : `voxel_size: 0.05`, `std_ratio: 1.0`

**Questions d'analyse :**
- Combien de points avant/après filtrage ?
- Quel filtre réduit le plus le nombre de points ?
- Quels paramètres donnent le meilleur compromis qualité/densité pour votre environnement ?

---
## Section 8 : Questions de Synthèse du laboratoire 2

Répondez aux questions suivantes pour valider votre compréhension :

### Question 1 : Actions vs Services

**Pourquoi utiliser une action plutôt qu'un service pour StartScan ?**


---

### Question 2: Executor et Callback Groups

**Expliquez pourquoi `lidar_traitement.py` a besoin à la fois de `MultiThreadedExecutor` ET de `ReentrantCallbackGroup`.**

Votre réponse doit mentionner :
- Ce qui se passerait sans `MultiThreadedExecutor` (deadlock dans WAITING_SERVO)
- Ce qui se passerait sans `ReentrantCallbackGroup` (annulation impossible)
- Le rôle de chacun

---

### Question 3 : Paramétrage

**Quel est l'avantage d'avoir les paramètres de filtrage dans le Goal de l'action plutôt que comme [paramètres ROS2](https://docs.ros.org/en/rolling/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Parameters/Understanding-ROS2-Parameters.html) du node ?**

---

### Question 4

Dans le fichier `/home/ubuntu/ros2_ws/src/jerro/jerro_description/urdf/jerro.urdf`, les axes du joint `base_link_to_servomotor` sont x=0, y=-1 et z=0. A partir du fichier URDF et de la [datasheet du lidar](https://download-en.slamtec.com/api/download/rplidar-s2m1-Rx-datasheet/2.3?lang=en) expliquez ces valeurs.

```xml
 <joint name="base_link_to_servomotor" type="revolute">
    <parent link="base_link" />
    <child link="servomotor" />
    <origin xyz="0.088 0 0.054" rpy="0 0 0" />
    <axis xyz="0 -1 0"/>
    <limit effort="1000.0" lower="-1" upper="1" velocity="0.5"/>  
    </joint>
```



---

### Question 5 : Ordre des filtres

**Expliquez pourquoi on applique le crop en premier et le voxel downsampling en dernier.**



---

### Question 6 : Paramètres statistiques

**Un `nb_neighbors` trop petit dans le filtre statistique risque d'éliminer des points valides. Pourquoi ?**


---

### Question 7

**Implémentez les 3 filtres tels que présentés. Présenter des images pour montrer la différence entre les filtres**

---

### Question 8
**Pour utiliser la caméra avec le Raspberry, des modifications dans certains fichiers ont dû être réalisées. À partir du readme du package [v4l2_camera](https://docs.ros.org/en/rolling/p/v4l2_camera/), expliquer quelles ont été ces modifications et pourquoi.**

---

### Question 9

**En observant les messages des topics `image_raw` et `image_raw/compressed`, expliquer la différence entre ces deux types de messages et ce que leur utilisation implique**

---

### Question 10
**Pour utiliser la caméra, un fichier de configuration a été placé dans le dossier `/home/ubuntu/.ros/camera_info/ost.yaml`. Expliquer les différents paramètres de ce fichier et quelle est son utilité. Aussi, quelles seraient les conséquences de son absence**


---
## Démonstration au chargé de laboratoire

À la fin de ce laboratoire, vous devez démontrer :

### Checklist de validation

- [ ] **Code compilé sans erreur**
- [ ] **État PROCESSING ajouté** à la machine à états
- [ ] **Trois filtres implémentés** (crop, statistical, voxel)
- [ ] **Transition conditionnelle** fonctionne (filtrage activé/désactivé)
- [ ] **Logs affichent** le nombre de points avant/après chaque filtre
- [ ] **Visualisation RViz** montre la différence avant/après filtrage
- [ ] **Paramètres optimisés** pour votre environnement
- [ ] **Questions de synthèse** répondues

### Bonus (optionnel)

- [ ] **Radius outlier removal** implémenté et testé
- [ ] **Comparaison quantitative** des différentes configurations (tableau de résultats)
- [ ] **Analyse de performance** (temps d'exécution de chaque filtre)

---
## Ressources utiles

### Documentation Open3D

- **Page principale** : https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html
- **Statistical Outlier Removal** : https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#statistical-outlier-removal
- **Radius Outlier Removal** : https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#radius-outlier-removal
- **Downsampling** : https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#downsampling
- **Bounding Volume Cropping** : https://www.open3d.org/html/tutorial/Advanced/pointcloud_outlier_removal.html#bounding-volume-cropping

### Documentation ROS2

- **Actions** : https://docs.ros.org/en/rolling/Tutorials/Beginner-CLI-Tools/Understanding-ROS2-Actions/Understanding-ROS2-Actions.html
- **Action Servers (Python)** : https://docs.ros.org/en/rolling/Tutorials/Intermediate/Writing-an-Action-Server-Client/Py.html
- **Executors et Callback Groups** : https://docs.ros.org/en/rolling/Concepts/Intermediate/About-Executors.html

### Code de référence

- `lidar_traitement.py` : Version avec Action Server, `while True`, `MultiThreadedExecutor`, `ReentrantCallbackGroup`
- `lidar_traitement_v1.py` : Version précédente avec services et timer (pour comparaison)